# Third Attempt

In [24]:
import torch
import numpy as np

## Load/ Preprocess Data

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = "physionet.org/files/ptb-xl/1.0.3/"
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()
sub_classes = agg_df.diagnostic_subclass.unique().tolist()
super_classes = agg_df.diagnostic_class.unique().tolist()

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel_index(class_list):
    return np.array([classes.index(cls) for cls in class_list])

# encodes the labeled as a 23d vector of 1's and zeros (based off of sub classes list)
def encode_multilabel_sub(class_list):
    return np.array([1 if cls in class_list else 0 for cls in sub_classes])

# encodes the labeled as a 5d vector of 1's and zeros (based off of sub classes list)
def encode_multilabel_super(class_list):
    return np.array([1 if cls in class_list else 0 for cls in super_classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    argmaxs = []
    max_prct = 0
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(key) #diagnostic, diagnostic_subclass #agg_df.loc[key].index
            if y_dic[key] > max_prct:
                if argmaxs != []:
                    argmaxs.pop(0)
                argmaxs.append(key)
    return list(set(tmp))
    return list(set(tmp)), list(set(argmaxs))

def aggregate_diagnostic_max(y_dic):
    tmp = []
    max_prct = 0
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] >= max_prct:
                tmp.append(key)
                max_prct = y_dic[key]
    return list(set(tmp))

def aggregate_diagnostic_sub(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(agg_df.loc[key].diagnostic_subclass)
    return list(set(tmp))

def aggregate_diagnostic_super(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['indv_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)
Y['indv_diagnostic_str'] = str(Y['indv_diagnostic'])
Y['indv_diagnostic_max'] = Y.scp_codes.apply(aggregate_diagnostic_max).apply(encode_multilabel_index)

Y['sub_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic_sub).apply(encode_multilabel_sub)
Y['sub_diagnostic_str'] = str(Y['sub_diagnostic'])

Y['super_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic_super).apply(encode_multilabel_super)
Y['super_diagnostic_str'] = str(Y['super_diagnostic'])




KeyboardInterrupt: 

In [ ]:
Y['indv_diagnostic']

ecg_id
1        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
5        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                               ...                        
21833    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
21834    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
21835    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
21836    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
21837    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: indv_diagnostic, Length: 21799, dtype: object

In [ ]:
Y['indv_diagnostic_max']

ecg_id
1         [4]
2         [4]
3         [4]
4         [4]
5         [4]
         ... 
21833     [0]
21834     [4]
21835    [25]
21836     [4]
21837     [4]
Name: indv_diagnostic_max, Length: 21799, dtype: object

## Data Prep

In [ ]:
from sklearn.model_selection import train_test_split

#stratified sample

Y_transformed = np.stack(Y.indv_diagnostic.values).astype(np.float32)

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, Y_transformed, Y.index.to_numpy(), test_size=0.1, random_state=56) #stratify=Y_transforme

In [ ]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [ ]:
max_dig = Y.loc[idx_train]['indv_diagnostic_max']

In [ ]:
from torch.utils.data import WeightedRandomSampler

condition_freq = y_train_torch.sum(axis=0)
condition_inv_freq = 1.0 / np.log(condition_freq + 1e-6)

row_weights = (y_train_torch * condition_inv_freq).sum(axis=1)
row_weights = row_weights / row_weights.sum()
sampler = WeightedRandomSampler(
    weights=row_weights,
    num_samples=len(row_weights),
    replacement = True
)

/tmp/ipykernel_2124/893458240.py:4: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  condition_inv_freq = 1.0 / np.log(condition_freq + 1e-6)


In [ ]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, sampler=sampler)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


In [ ]:
# train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
# train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

# test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
# test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    #starting with 96

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
    #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
    #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
    #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [ ]:
def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()

    #pos_weight = torch.tensor([5.0]).to(device)
    #loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    #condition_freq = y_train.sum(axis=0)
    #pos_weight = (len(y_train) - condition_freq) / condition_freq

    f = torch.tensor(y_train).sum(dim=0)
    N = y_train.shape[0]
    pos_weight = (N - f) / (f + 1e-6)
    pos_weight = torch.log(pos_weight.to(device))
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
    #loss_fn = torch.nn.BCEWithLogitsLoss()

    lr = 1e-3
    #opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            #print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
        scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [ ]:
y_train.sum(axis=0)

array([1.649e+03, 4.850e+02, 1.240e+02, 9.800e+01, 8.051e+03, 1.024e+03,
       1.448e+03, 1.066e+03, 1.439e+03, 1.043e+03, 1.001e+03, 7.020e+02,
       6.980e+02, 4.730e+02, 4.850e+02, 4.740e+02, 3.060e+02, 3.000e+02,
       1.120e+02, 1.900e+02, 1.550e+02, 1.720e+02, 5.300e+01, 1.130e+02,
       1.580e+02, 1.390e+02, 1.170e+02, 1.060e+02, 4.800e+01, 1.100e+01,
       7.600e+01, 1.400e+01, 6.000e+01, 6.900e+01, 2.800e+01, 3.700e+01,
       2.800e+01, 1.800e+01, 1.400e+01, 1.300e+01, 2.000e+00, 1.100e+01,
       1.300e+01, 1.300e+01], dtype=float32)

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class ResidualBlock1D(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=9, dropout=0.1):
    super().__init__()
    padding = kernel_size // 2

    self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
    self.bn1 = nn.BatchNorm1d(out_channels)

    self.conv12 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
    self.bn2 = nn.BatchNorm1d(out_channels)

    self.dropout = nn.Dropout(dropout)

    if in_channels != out_channels:
      #residual would be uneven here
      self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size=1) # reduce the channels
    else:
      self.shortcut = nn.Identity()

  def forward(self, x):
      identity = self.shortcut(x) #residual

      x = self.conv1(x)
      x = self.bn1(x)
      x = F.relu(x)

      x = self.dropout(x)

      x = self.conv12(x)
      x = self.bn2(x)
      
      x = x + identity

      x = F.relu(x)

      return x
    

class CINet(nn.Module):
  def __init__(self):
    super(CINet,self).__init__()
    self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
    
    self.block1 = ResidualBlock1D(12, 64, kernel_size=21, dropout=0.1)

    self.block2 = ResidualBlock1D(64, 128, kernel_size=21, dropout=0.1)

    self.block3 = ResidualBlock1D(128, 256, kernel_size=21, dropout=0.2)

    self.block4 = ResidualBlock1D(256, 512, kernel_size=21, dropout=0.2)

    self.gap = nn.AdaptiveAvgPool1d(1)
    
    self.drop_fc = nn.Dropout(0.4)
    self.fc1 = nn.Linear(512, 128)
    self.fc2 = nn.Linear(128, 44)

  def forward(self, x):

    x = self.block1(x)
    x = self.pool(x)

    x = self.block2(x)
    x = self.pool(x)

    x = self.block3(x)
    x = self.pool(x)

    x = self.block4(x)

    x = self.gap(x)
    x = x.squeeze(-1)

    x = self.drop_fc(x)
    x = F.relu(self.fc1(x))
    x = self.drop_fc(x)
    
    x = self.fc2(x)
    return x
    

"""
class CINet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    
    self.drop_conv = nn.Dropout(0.2)
    self.drop_fc = nn.Dropout(0.4)

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.gap = nn.AdaptiveAvgPool1d(1)
    
    self.fc1 = nn.Linear(512, 128)
    self.fc2 = nn.Linear(128, 44)

  def forward(self, x):

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.drop_conv(x)

    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.drop_conv(x)

    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = self.drop_conv(x)

    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = self.gap(x)
    x = x.squeeze(-1)
    x = x.drop_fc(x)
    x = F.relu(self.fc1(x))
    x = x.drop_fc(x)
    x = self.fc2(x)
    return x"""

'\nclass CINet(nn.Module):\n  def __init__(self):\n    super(CNet,self).__init__()\n\n    self.drop_conv = nn.Dropout(0.2)\n    self.drop_fc = nn.Dropout(0.4)\n\n    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)\n    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)\n    self.bn1 = nn.BatchNorm1d(64)\n\n    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)\n\n    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)\n    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)\n    self.bn2 = nn.BatchNorm1d(128)\n\n    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)\n    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)\n    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)\n    self.bn3 = nn.BatchNorm1d(256)\n\n    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)\n    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)\n    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)\n    self.bn4 = nn.BatchNorm1d(512)\n\n    self.gap = nn.AdaptiveAvgPool1d(1)\n\n    self.fc1 = nn.Linear(512, 128)\n    self.fc2 = nn.Linear(

In [ ]:
inet = CINet().to("cuda")
train_model(inet, train_dataloader, test_dataloader, 30, 1e-3)

/tmp/ipykernel_2124/2018850685.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))


Epoch 0 train loss: 0.29295433870841314
Epoch 1 train loss: 0.26599532751675925
Epoch 2 train loss: 0.25313262316225404
Epoch 3 train loss: 0.25073810212969394
Epoch 4 train loss: 0.2483514905072967
Epoch 5 train loss: 0.23852888198648292
Epoch 6 train loss: 0.23635891431803036
Epoch 7 train loss: 0.23399062413443183
Epoch 8 train loss: 0.23351189042729742
Epoch 9 train loss: 0.2278468274186799
Epoch 10 train loss: 0.22892953433606059
Epoch 11 train loss: 0.22577944422195323
Epoch 12 train loss: 0.22251913599451512
Epoch 13 train loss: 0.22192592297586633
Epoch 14 train loss: 0.2228244941572413
Epoch 15 train loss: 0.2198939413705944
Epoch 16 train loss: 0.21583974743656303
Epoch 17 train loss: 0.2112965415374464
Epoch 18 train loss: 0.21355452009928733
Epoch 19 train loss: 0.20920778138726853
Epoch 20 train loss: 0.20813613136204911
Epoch 21 train loss: 0.2034956504157001
Epoch 22 train loss: 0.20253577245951476
Epoch 23 train loss: 0.1996533332410775
Epoch 24 train loss: 0.1996511116

In [ ]:
def my_jaccard_score(y_pred, y_true):
    intersection = ((y_pred == 1) & (y_true == 1)).sum()
    union = ((y_pred == 1) | (y_true == 1)).sum()

    if union == 0:
        return 0.0
    
    return intersection / union


def macro_precision(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_p = ((y_pred == 1) & (y_true == 0)).sum(dim=0)
    return t_p.float() / (t_p + f_p).float().clamp(min=1e-8)

def macro_recall(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_n = ((y_pred == 0) & (y_true == 1)).sum(dim=0)
    return t_p.float() / (t_p + f_n).float().clamp(min=1e-8)

def macro_f1(y_pred, y_true):
    m_p = macro_precision(y_pred, y_true)
    m_r = macro_recall(y_pred, y_true)
    return 2 * m_p * m_r / (m_p + m_r).clamp(min=1e-8)

from sklearn.metrics import jaccard_score

@torch.no_grad()
def dig_accurate(y_preds, y_ground):
    y_preds_cpu = y_preds.detach().to("cpu", dtype=torch.int32)
    y_ground_cpu = y_ground.detach().to("cpu", dtype=torch.int32)
    element_wise_eq = y_preds_cpu == y_ground_cpu
    y_preds = y_preds_cpu.numpy()
    y_ground = y_ground_cpu.numpy()

    row_acc = torch.all(element_wise_eq, dim=1).float().mean().item()
    elem_acc = element_wise_eq.float().mean().item()
    print(f"Jaccard_Score: {jaccard_score(y_ground, y_preds, average="samples")}")
    print(f"Row-wise Accuracy (perfect match): {row_acc}")
    print(f"Element Wise Accuracy: {elem_acc}")

In [ ]:
inet.eval()
out = (torch.sigmoid(inet(X_test_torch.to("cuda"))) > 0.5)
input = y_test_torch.to("cuda")
print(dig_accurate(out.int(), input.int()))

Jaccard_Score: 0.44534567496723465
Row-wise Accuracy (perfect match): 0.4192660450935364
Element Wise Accuracy: 0.9752501845359802
None


/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
inet.eval()

with torch.no_grad():
    logits = inet(X_test_torch.to("cuda"))
    probs = torch.sigmoid(logits)

input = y_test_torch.to("cuda").int()

best_thresh = None
best_score = -1

for t in torch.arange(0.0, 1.01, 0.05, device="cuda"):
    preds = (probs > t).int()

    print(f"\nThreshold: {t.item():.2f}")
    dig_accurate(preds, input)


Threshold: 0.00
Jaccard_Score: 0.025594245204336948
Row-wise Accuracy (perfect match): 0.0
Element Wise Accuracy: 0.02559424564242363

Threshold: 0.05
Jaccard_Score: 0.14861129342861695
Row-wise Accuracy (perfect match): 0.0009174311999231577
Element Wise Accuracy: 0.7680566906929016

Threshold: 0.10
Jaccard_Score: 0.2625489542471026
Row-wise Accuracy (perfect match): 0.09449541568756104
Element Wise Accuracy: 0.8484570384025574

Threshold: 0.15
Jaccard_Score: 0.3525589566985723
Row-wise Accuracy (perfect match): 0.18853211402893066
Element Wise Accuracy: 0.8915762901306152

Threshold: 0.20
Jaccard_Score: 0.4072310637395351
Row-wise Accuracy (perfect match): 0.25229358673095703
Element Wise Accuracy: 0.919422447681427

Threshold: 0.25
Jaccard_Score: 0.4388221061558218
Row-wise Accuracy (perfect match): 0.2944954037666321
Element Wise Accuracy: 0.9371872544288635

Threshold: 0.30
Jaccard_Score: 0.46660923492918904
Row-wise Accuracy (perfect match): 0.3412843942642212
Element Wise Accur

/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [ ]:
inet.eval()
count = 0
j_score = 0.0
f1_score = 0.0
p_score = 0.0
r_score = 0.0

a_m_f1_score = []
a_m_p_score = []
a_m_r_score = []


for i, (x_test, y_test) in enumerate(test_dataloader):
    # if i > 2: 
    #     break
    val = (torch.sigmoid(inet(x_test.to("cuda"))) > 0.35).int()
    count += 1
    y_test = y_test.to("cuda").int()
    j_score += my_jaccard_score(val, y_test)
    a_m_f1_score.append(macro_f1(val, y_test))
    a_m_p_score.append(macro_precision(val, y_test))
    a_m_r_score.append(macro_recall(val, y_test))

a_m_f1_score = torch.stack(a_m_f1_score).cpu()
a_m_p_score = torch.stack(a_m_p_score).cpu()
a_m_r_score = torch.stack(a_m_r_score).cpu()
print(a_m_f1_score)
print(a_m_p_score)
print(a_m_r_score)
print(a_m_f1_score.mean(), a_m_p_score.mean(), a_m_r_score.mean())

tensor([[0.4000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.2857, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.1818, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.5000, 0.0000, 1.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])
tensor([[0.2857, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.1667, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.1111, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.3333, 0.0000, 1.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])
tensor([[0.6667, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.5000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,